# Clase 213 — Kafka local + producer/consumer + snippets Kinesis

Notebook declarativo + simulación en proceso (no requiere Kafka). Para el demo real: levantar el `docker-compose.yml` de la celda 1.

## 1. docker-compose Kafka (KRaft, sin Zookeeper)

In [ ]:
compose = '''\
services:
  kafka:
    image: bitnami/kafka:3.7
    ports: ["9092:9092"]
    environment:
      KAFKA_CFG_NODE_ID: 1
      KAFKA_CFG_PROCESS_ROLES: controller,broker
      KAFKA_CFG_LISTENERS: PLAINTEXT://:9092,CONTROLLER://:9093
      KAFKA_CFG_ADVERTISED_LISTENERS: PLAINTEXT://localhost:9092
      KAFKA_CFG_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CFG_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_CFG_CONTROLLER_QUORUM_VOTERS: 1@kafka:9093
      KAFKA_CFG_AUTO_CREATE_TOPICS_ENABLE: "true"
    healthcheck:
      test: kafka-topics.sh --bootstrap-server localhost:9092 --list
      interval: 5s

  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    ports: ["8080:8080"]
    environment:
      KAFKA_CLUSTERS_0_NAME: local
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: kafka:9092
    depends_on: { kafka: { condition: service_healthy } }
'''
print(compose)
print('# levantar: docker-compose up -d')
print('# UI: http://localhost:8080')

## 2. Producer Python

In [ ]:
producer_src = '''\
# producer.py — pip install confluent-kafka faker
from confluent_kafka import Producer
from faker import Faker
import json, time, random

fake = Faker()
p = Producer({
    "bootstrap.servers": "localhost:9092",
    "acks": "all",                  # waitall replicas
    "enable.idempotence": True,     # no dup en retries
    "linger.ms": 10,                # batchear hasta 10ms
})

def delivery(err, msg):
    if err: print(f"❌ {err}")

for i in range(1000):
    user_id = f"user_{random.randint(1, 100)}"
    event = {"page": fake.uri_path(), "ts": time.time(), "i": i}
    p.produce(
        "clicks",
        key=user_id.encode(),           # mismo user → misma partition
        value=json.dumps(event).encode(),
        callback=delivery,
    )
    if i % 100 == 0: p.poll(0)
p.flush(timeout=10)
print("1000 mensajes enviados.")
'''
print(producer_src)

## 3. Consumer Python (at-least-once con commit manual)

In [ ]:
consumer_src = '''\
# consumer.py
from confluent_kafka import Consumer, KafkaError
import json, duckdb

con = duckdb.connect("clicks.duckdb")
con.execute("""
    CREATE TABLE IF NOT EXISTS clicks (
        user_id TEXT, page TEXT, ts DOUBLE, i INT,
        PRIMARY KEY (user_id, i)          -- idempotent: dedupe a nivel de DB
    )
""")

c = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "clicks-to-duckdb",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,         # commit manual = at-least-once
})
c.subscribe(["clicks"])

try:
    while True:
        msg = c.poll(timeout=1.0)
        if msg is None: continue
        if msg.error():
            if msg.error().code() != KafkaError._PARTITION_EOF: print(msg.error())
            continue
        user_id = msg.key().decode()
        event = json.loads(msg.value())
        con.execute("INSERT OR IGNORE INTO clicks VALUES (?, ?, ?, ?)",
                    [user_id, event["page"], event["ts"], event["i"]])
        c.commit(msg)   # commit DESPUÉS de procesar
finally:
    c.close()
    con.close()
'''
print(consumer_src)

## 4. Simulación in-process (sin Kafka real)

In [ ]:
from collections import defaultdict
import hashlib, json, time, random

class InMemoryTopic:
    def __init__(self, n_partitions=4):
        self.partitions = defaultdict(list)
        self.n = n_partitions
    def produce(self, key, value):
        p = int(hashlib.md5(key.encode()).hexdigest(), 16) % self.n
        self.partitions[p].append((key, value))
        return p

topic = InMemoryTopic(n_partitions=4)
for i in range(1000):
    key = f'user_{random.randint(1, 100)}'
    topic.produce(key, {'page': '/foo', 'i': i})

for p in range(topic.n):
    print(f'partition {p}: {len(topic.partitions[p]):>4} mensajes')

# Verificar: mismo user siempre va a la misma partition
u = 'user_42'
ps = set()
for _ in range(50):
    ps.add(topic.produce(u, {'i': 0}))
print(f'\nuser_42 fue a partitions: {ps} (debe ser solo 1)')

## 5. Kinesis equivalent

In [ ]:
kinesis = '''\
# Kinesis usa nombres distintos pero mismo modelo: shard ≈ partition.
import boto3, json
k = boto3.client("kinesis")

# Producer:
k.put_record(
    StreamName="clicks",
    Data=json.dumps({"page": "/foo"}).encode(),
    PartitionKey="user_42",   # → shard determinístico por hash de key
)

# Consumer enhanced fan-out (recommended):
shards = k.describe_stream(StreamName="clicks")["StreamDescription"]["Shards"]
for s in shards:
    iter_id = k.get_shard_iterator(
        StreamName="clicks", ShardId=s["ShardId"], ShardIteratorType="TRIM_HORIZON",
    )["ShardIterator"]
    out = k.get_records(ShardIterator=iter_id, Limit=10)
    for r in out["Records"]:
        print(json.loads(r["Data"]))

# Producción: usar KCL (Kinesis Client Library) — checkpoint en DynamoDB.
'''
print(kinesis)

## Ejercicio guiado

1. Levantá el docker-compose Kafka. Creá topic `clicks` con `--partitions 4`.
2. Corré el producer y 2 consumers en el mismo group. Confirmá reparto 2-2 de partitions.
3. Matá un consumer mid-run. Observá rebalancing en el otro (toma las 4).
4. Forzá lag: producer escribe 10× más rápido que consumer puede procesar. Usá `kafka-consumer-groups --describe` para ver lag.
5. Bonus: integrá con Flink o Spark Structured Streaming para hacer aggregations rolling.

## Conclusiones

- Topic + partitions + offsets = el modelo mental fundamental; Kinesis/Pub/Sub son re-skins.
- Key → partition determinístico → orden por key garantizado, paralelismo entre keys.
- At-least-once + dedupe en DB resuelve el 95% de los casos sin pagar exactly-once overhead.
- Consumer lag es la métrica de salud — alertar si crece sin techo.